# Mean vs. median for per-condition ion intensity: impact on epsilon

`QuantScoresHYE.compute_condition_stats` aggregates the log2 intensity of a precursor ion across
the replicate raw files of a condition using the **mean** (`proteobench/score/quantscoresHYE.py`,
`log_Intensity_mean_A`/`log_Intensity_mean_B`). This value feeds directly into
`log2_A_vs_B = log_Intensity_mean_A - log_Intensity_mean_B`, which in turn is the basis for
`epsilon = log2_A_vs_B - log2_expectedRatio` (accuracy) and the precision epsilons.

This notebook checks how much `epsilon` (and quantification accuracy in general) would change if
that per-condition aggregation used the **median** of the replicate log2 intensities instead of the
mean, everything else in the pipeline held constant.

This version uses **every real, public submission** for the `quant_lfq_DDA_ion_QExactive` module,
and mimics the website's actual main benchmarking plot with its **default settings**:
- **Median** aggregation of the per-precursor `epsilon` values (vs. mean)
- **Species-weighted** mode (`eq_species`): average the per-species medians with equal weight
  regardless of precursor count (vs. "global", which pools all species as one population). This
  is `LFQHYEPlotGenerator.plot_main_metric`'s actual default mode.
- **k = 3**: only precursors observed in at least 3 replicate raw files count
  (`MIN_NR_OBSERVED`, matching `QuantDatapointHYE`'s `default_cutoff_min_feature` and the site's
  "minimum number of quantification values per raw file" slider)

All public datapoints are pulled from the
[`Proteobench/Results_quant_ion_DDA`](https://github.com/Proteobench/Results_quant_ion_DDA) GitHub
repo, and each submission's original uploaded input file is downloaded from the public
intermediate-data server (`https://proteobench.cubimed.rub.de/datasets/`). Tools differ widely in
how many public submissions they have (e.g. many more MaxQuant/i2MassChroQ runs than
Sage/PEAKS/quantms at the time of writing), so results are reported both per individual submission
and pooled per tool.

Approach:
1. Fetch every public datapoint for the module.
2. Download each submission's original input file (`input_file.*` inside its public data zip).
3. Parse it through the normal ProteoBench pipeline up to the replicate-level intensity table
   (`relevant_columns_df`).
4. Compute `epsilon` the normal (mean-based) way with the existing `QuantScoresHYE.compute_condition_stats`.
5. Compute `epsilon` again with a median-based drop-in replacement of that same function.
6. Keep only precursors with `nr_observed >= 3`, exactly like the website's default view.
7. Validate the mean-based recomputation against the actual stored production values, then compare
   the two `epsilon` aggregations per precursor, per submission, per tool (pooling all of a tool's
   submissions), and per species.

All figures in this notebook are built with Plotly, styled to match the ProteoBench manuscript
figures (`Astral_diaPASEF_main_figure.ipynb`, `Astral_in_depth_figures.ipynb`): the same tool color
palette (`LFQHYEPlotGenerator.software_colors`), the module's own species colors, and a shared
white/Arial/black-axis-line layout.

This requires network access (GitHub + the ProteoBench public data server) and downloads/parses
every public submission for the module, so it takes noticeably longer to run than looking at a
single submission per tool.


In [19]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from proteobench.io.parsing.parse_ion import load_input_file
from proteobench.io.parsing.parse_settings import ParseSettingsBuilder
from proteobench.score.quantscoresHYE import QuantScoresHYE
from proteobench.utils.server_io import get_merged_json, get_raw_data

warnings.filterwarnings("ignore")  # silence known "0 intensity rows removed" / proforma parser warnings

MODULE_ID = "quant_lfq_DIA_ion_Astral"
RESULTS_REPO_URL = "https://github.com/Proteobench/Results_quant_ion_DIA_Astral/archive/refs/heads/main.zip"
PARSE_SETTINGS_DIR = os.path.join(
    "..","..", "ProteoBench", "proteobench", "io", "parsing", "io_parse_settings", "Quant", "lfq", "DIA", "ion", "Astral"
)
RAW_DATA_DIR = "post_analysis/quant/lfq/ion/dia/astral_diaPasef/extracted_files"  # gitignored; see jupyter_notebooks/.gitignore
PRECURSOR = "precursor ion"

# Cache for the expensive per-submission extraction loop below (compare_tool over every public
# submission -- the slow, network+parsing-heavy step). Set OVERWRITE_CACHE=True to force a full
# re-extraction (e.g. after new public submissions land).
CACHE_PATH = "/mnt/c/Users/robbe/Work/Post-processing/analysis/all_datapoints_Astral.pkl"
OVERWRITE_CACHE = False

# Match the website's default main-plot settings exactly:
# - MIN_NR_OBSERVED (k): QuantDatapointHYE.generate_datapoint's default_cutoff_min_feature.
#   Only precursors observed in >= k replicate raw files are counted ("minimum number of
#   quantification values per raw file" slider on the site, default 3).
# - METRIC_AGG: "median" -> median absolute epsilon (vs. "mean").
# - METRIC_MODE: "eq_species" -> Species-weighted: average the per-species medians with equal
#   weight regardless of precursor count (vs. "global", which pools all species as one
#   population). This is LFQHYEPlotGenerator.plot_main_metric's actual default mode.
MIN_NR_OBSERVED = 3
METRIC_AGG = "median"
METRIC_MODE = "eq_species"


## ProteoBench-styled Plotly helpers

Same tool colors, display names, and layout constants as the manuscript figures
(`Astral_diaPASEF_main_figure.ipynb`, `Astral_in_depth_figures.ipynb`), plus a species -> color
lookup taken directly from the module's own settings so in-plot species colors match the
website's in-depth plots. `eq_species_metric` mirrors
`QuantDatapointHYE.get_epsilon_metrics()`'s "eq_species" aggregation: per-species
median/mean-of-abs-value, then a plain (equal-weight) mean across species.


In [36]:
# Same tool -> color mapping as LFQHYEPlotGenerator.software_colors / the manuscript figures, so
# plots here match the website's and the manuscript's colors exactly.
_SOFTWARE_COLORS = {
    "AlphaDIA": "#1D2732",
    "DIA-NN": "#999934",
    "Spectronaut": "#007548",
    "FragPipe (DIA-NN quant)": "#F89008",
    "PEAKS": "#f032e6",
    "MaxQuant": "#88ccef",
}
_DISPLAY_NAMES = {"FragPipe (DIA-NN quant)": "FragPipe"}


def _display(tool):
    return _DISPLAY_NAMES.get(tool, tool)


_BASE_LAYOUT = dict(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=15, color="black"),
)
_AXIS_STYLE = dict(
    showline=True,
    linecolor="black",
    linewidth=1,
    mirror=True,
    gridcolor="lightgray",
    gridwidth=1,
    title_font=dict(size=20)
)
_LEGEND_STYLE = dict(
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="lightgray",
    borderwidth=1,
)


def _hex_to_rgba(hex_color, alpha=0.25):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


# Species -> color, taken from the module's own settings (species_expected_ratio). The
# species/color mapping is per-module (not per-tool), so any tool the module supports works here.
_species_expected_ratio = (
    ParseSettingsBuilder(parse_settings_dir=PARSE_SETTINGS_DIR, module_id=MODULE_ID)
    .build_parser("DIA-NN")
    .species_expected_ratio()
)
_SPECIES_COLORS = {sp: d["color"] for sp, d in _species_expected_ratio.items()}


def eq_species_metric(df, value_col, agg=METRIC_AGG):
    """Equal-weighted species average of |value_col|: per-species agg (median/mean of the
    absolute value), then a plain mean across species -- matches
    QuantDatapointHYE.get_epsilon_metrics()'s "eq_species" mode exactly."""
    agg_func = (lambda s: s.abs().median()) if agg == "median" else (lambda s: s.abs().mean())
    return df.groupby("species")[value_col].apply(agg_func).mean()


## Median-based drop-in replacement

Mirrors `QuantScoresHYE.compute_condition_stats` exactly, except the per-precursor,
per-condition aggregation across replicates uses `median` instead of `mean` for the
log2 intensity. Everything upstream (summing intensities per raw file, filtering on
`min_intensity`) and downstream (`compute_epsilon`) is reused unchanged.


In [21]:
def compute_condition_stats_median(relevant_columns_df, min_intensity=0, precursor=PRECURSOR):
    """Same as QuantScoresHYE.compute_condition_stats, but aggregates log2 intensity
    across replicates with the median instead of the mean."""
    df = relevant_columns_df[relevant_columns_df["Intensity"] > min_intensity]

    quant_raw_df_int = (
        df.groupby([precursor, "Raw file", "Condition"])["Intensity"].agg(Intensity="sum", Count="size").reset_index()
    )
    quant_raw_df_int["log_Intensity"] = np.log2(quant_raw_df_int["Intensity"])

    quant_raw_df_count = quant_raw_df_int.groupby([precursor]).agg(nr_observed=("Raw file", "size"))

    quant_raw_df = (
        quant_raw_df_int.groupby([precursor, "Condition"])
        .agg(log_Intensity_median=("log_Intensity", "median"))
        .reset_index()
    )
    quant_raw_df = quant_raw_df.pivot(index=precursor, columns="Condition", values=["log_Intensity_median"]).reset_index()
    quant_raw_df.columns = [f"{x[0]}_{x[1]}" if len(str(x[1])) > 0 else x[0] for x in quant_raw_df.columns]

    # Same downstream column name ("log2_A_vs_B") as the mean-based path, so compute_epsilon can be reused as-is.
    quant_raw_df["log2_A_vs_B"] = quant_raw_df["log_Intensity_median_A"] - quant_raw_df["log_Intensity_median_B"]

    quant_raw_df = pd.merge(quant_raw_df, quant_raw_df_count, on=precursor, how="inner")
    return quant_raw_df


## Select all public submissions

Fetch every public datapoint for `quant_lfq_DDA_ion_QExactive` from the results repo. Each
submission's raw data zip (containing the original `input_file.*` the user uploaded) is then
downloaded and extracted via `proteobench.utils.server_io.get_raw_data`.


In [22]:
all_datapoints = get_merged_json(repo_url=RESULTS_REPO_URL)
print(f"{len(all_datapoints)} public datapoints for {MODULE_ID}")
print(all_datapoints["software_name"].value_counts())

submissions = all_datapoints[["id", "software_name", "intermediate_hash"]].reset_index(drop=True)
submissions


Combined 69 JSON files into 'combined_results.json'.
69 public datapoints for quant_lfq_DIA_ion_Astral
software_name
DIA-NN                     32
AlphaDIA                   15
Spectronaut                 8
PEAKS                       8
FragPipe (DIA-NN quant)     5
MaxQuant                    1
Name: count, dtype: int64


,id,software_name,intermediate_hash
0,AlphaDIA_20250704_110414,AlphaDIA,059a69e4eccf183869f956c4c9494d31945f6849
1,AlphaDIA_20250605_170349,AlphaDIA,11d1e3cad24d35b31110e82274bd0fef65435a34
2,AlphaDIA_20251017_120515,AlphaDIA,1bd69ca0cc5060987715afa2933b54f61fd1522e
3,FragPipe (DIA-NN quant)_20251027_114904,FragPipe (DIA-NN quant),1cd54fe0ba5d6c98ef47688414c4f4c409a6eecf
4,DIA-NN_20260805_082913,DIA-NN,200c98108a367991eb6d1b4cf867d5a617d62f10
...,...,...,...
64,DIA-NN_20251113_143358,DIA-NN,deb0844196e184b7eccd1c77487a2e0d4dafdad1
65,DIA-NN_20260805_080817,DIA-NN,e5496185aa8686dffac336d3979d55e78e86226f
66,DIA-NN_20251027_121015,DIA-NN,f4cb5e763671291e8d7d463a5b01a5c2831a249a
67,FragPipe (DIA-NN quant)_20251027_120307,FragPipe (DIA-NN quant),fae9b73eb9ac763f4b7f6f2f0dfb60b62e209b13


In [23]:
hash_to_dir = get_raw_data(submissions, base_url="https://proteobench.cubimed.rub.de/datasets/", output_directory=RAW_DATA_DIR)

SUBMISSION_INPUT_FILES = {}  # submission id -> (tool, input_file_path, input_file_secondary_path_or_None)
for _, row in submissions.iterrows():
    sub_id, tool, h = row["id"], row["software_name"], row["intermediate_hash"]
    location = hash_to_dir.get(h)
    if location is None or not os.path.isdir(location):
        print(f"{sub_id}: no raw data directory found for hash {h}, skipping")
        continue

    # The raw data zip stores the primary file as "input_file.<ext>" and, for two-file tools like
    # AlphaDIA (precursor.matrix.tsv + precursors.tsv), a secondary file as
    # "input_file_secondary.<ext>" (see quant_base_module.py's write_intermediate_raw()). Both
    # need to be matched separately, since "input_file_secondary.*" also starts with "input_file".
    primary_matches = [
        f
        for f in os.listdir(location)
        if f.startswith("input_file")
        and not f.startswith("input_file_secondary")
        and os.path.isfile(os.path.join(location, f))
    ]
    secondary_matches = [
        f
        for f in os.listdir(location)
        if f.startswith("input_file_secondary") and os.path.isfile(os.path.join(location, f))
    ]
    if not primary_matches:
        print(f"{sub_id}: no input_file.* found in {location}, skipping")
        continue

    primary_path = os.path.join(location, primary_matches[0])
    secondary_path = os.path.join(location, secondary_matches[0]) if secondary_matches else None

    # AlphaDIA has two on-disk formats: v1 (two TSVs -- a long-format precursors.tsv with all
    # metadata, and a precursor.matrix.tsv with the per-run wide intensity columns) and v2 (a
    # single self-contained parquet, which _load_alphadia() pivots from long to wide internally
    # via its "raw.name" column if needed -- see parse_ion.py's _load_alphadia()). Only the v1
    # TSV pair genuinely requires both files: a lone v1 TSV that turns out to be the long-format
    # file has no per-run columns, so the downstream df.melt(..., value_vars=melt_vars) in
    # parse_settings.py raises a "value_vars not present" KeyError. A lone v2 parquet is fine.
    is_alphadia_v1_tsv = tool == "AlphaDIA" and not primary_path.lower().endswith(".parquet")
    if is_alphadia_v1_tsv and secondary_path is None:
        print(f"{sub_id} ({tool}): AlphaDIA v1 (TSV) submission is missing its secondary input file in {location}, skipping")
        continue

    SUBMISSION_INPUT_FILES[sub_id] = (tool, primary_path, secondary_path)

print(f"\nResolved input files for {len(SUBMISSION_INPUT_FILES)}/{len(submissions)} submissions")


Folder already exists and is not empty, skipping download: post_analysis/quant/lfq/ion/dia/astral_diaPasef/extracted_files/059a69e4eccf183869f956c4c9494d31945f6849
Folder already exists and is not empty, skipping download: post_analysis/quant/lfq/ion/dia/astral_diaPasef/extracted_files/11d1e3cad24d35b31110e82274bd0fef65435a34
Folder already exists and is not empty, skipping download: post_analysis/quant/lfq/ion/dia/astral_diaPasef/extracted_files/1bd69ca0cc5060987715afa2933b54f61fd1522e
Folder already exists and is not empty, skipping download: post_analysis/quant/lfq/ion/dia/astral_diaPasef/extracted_files/1cd54fe0ba5d6c98ef47688414c4f4c409a6eecf
Folder already exists and is not empty, skipping download: post_analysis/quant/lfq/ion/dia/astral_diaPasef/extracted_files/200c98108a367991eb6d1b4cf867d5a617d62f10
Folder already exists and is not empty, skipping download: post_analysis/quant/lfq/ion/dia/astral_diaPasef/extracted_files/2115d5e394c382dc1d18f7e12ce0809311d584af
Folder already e

## Run both aggregations for every submission (website-matching filter applied)

`compare_tool` reruns the standard parsing/standardization steps once on a downloaded input file,
then branches into the mean-based (production) and median-based `compute_condition_stats`, and
computes `epsilon` for both, keeping only precursors with `nr_observed >= MIN_NR_OBSERVED` (k=3) —
matching `QuantDatapointHYE.get_epsilon_metrics()`'s `df_slice = df[df["nr_observed"] >= min_nr_observed]`.

It returns **two different things**, because they have different natural populations:
- `n_precursors` / `median_abs_epsilon_*_based`: computed directly on the full (k-filtered)
  `epsilon_mean`/`epsilon_median` tables, exactly like the website. A precursor quantified in only
  one of conditions A/B has `epsilon = NaN` but still counts towards this population — `nr_feature`
  on the website counts it too, and `.median()` already skips NaN automatically.
- `comparison_df`: an inner join of the two epsilon series, restricted to precursors where **both**
  aggregations give a defined (non-NaN) value — i.e. quantified in both conditions. This is the
  correct (necessarily smaller) population for a per-precursor mean-vs-median *difference* analysis
  (`abs_diff`, correlation), which is meaningless when one side is undefined.

An earlier version of this notebook computed `n_precursors` from `comparison_df` directly, which
silently dropped every precursor quantified in only one condition and undercounted relative to the
website by up to ~36,000 precursors for some submissions (see the Validation section).

**Caching:** looping `compare_tool` over every public submission downloads and re-parses each raw
result file, which is the slow, network/CPU-heavy step in this notebook. The cell below caches its
four outputs (`combined`, `combined_epsilon_mean_k`, `combined_epsilon_median_k`,
`result_metadata`) to `CACHE_PATH` (`all_datapoints_Astral.pkl`) and loads from there on subsequent
runs instead of recomputing. Set `OVERWRITE_CACHE = True` in the settings cell to force a full
refresh (e.g. once new public submissions land).


In [24]:
def compare_tool(tool, input_file, input_file_secondary=None):
    input_df = load_input_file(input_file, tool, input_csv_secondary=input_file_secondary)

    builder = ParseSettingsBuilder(parse_settings_dir=PARSE_SETTINGS_DIR, module_id=MODULE_ID)
    parse_settings = builder.build_parser(tool)
    prepared_df, replicate_to_raw = parse_settings.convert_to_standard_format(input_df)

    relevant_columns_df = prepared_df[["Raw file", PRECURSOR, "Intensity"]].copy()
    replicate_to_raw_df = QuantScoresHYE.convert_replicate_to_raw(replicate_to_raw)
    relevant_columns_df = pd.merge(relevant_columns_df, replicate_to_raw_df, on="Raw file", how="inner")

    quant_mean_df = QuantScoresHYE.compute_condition_stats(relevant_columns_df, precursor=PRECURSOR)
    quant_median_df = compute_condition_stats_median(relevant_columns_df, precursor=PRECURSOR)

    species_prec_ion = list(parse_settings.species_dict().values()) + [PRECURSOR]
    prec_ion_to_species = prepared_df[species_prec_ion].drop_duplicates()

    mean_withspecies = pd.merge(quant_mean_df, prec_ion_to_species, on=PRECURSOR, how="inner")
    median_withspecies = pd.merge(quant_median_df, prec_ion_to_species, on=PRECURSOR, how="inner")

    # Full single-species tables, unfiltered by nr_observed (k=1 population).
    epsilon_mean = QuantScoresHYE.compute_epsilon(mean_withspecies, parse_settings.species_expected_ratio())
    epsilon_median = QuantScoresHYE.compute_epsilon(median_withspecies, parse_settings.species_expected_ratio())

    def k_stats(epsilon_df, k, mode=METRIC_MODE):
        sub = epsilon_df[epsilon_df["nr_observed"] >= k]
        if mode == "eq_species":
            return len(sub), eq_species_metric(sub, "epsilon")
        agg_func = (lambda s: s.abs().median()) if METRIC_AGG == "median" else (lambda s: s.abs().mean())
        return len(sub), agg_func(sub["epsilon"])

    n_k1, median_abs_epsilon_mean_based_k1 = k_stats(epsilon_mean, 1)
    _, median_abs_epsilon_median_based_k1 = k_stats(epsilon_median, 1)
    n_k3, median_abs_epsilon_mean_based_k3 = k_stats(epsilon_mean, MIN_NR_OBSERVED)
    _, median_abs_epsilon_median_based_k3 = k_stats(epsilon_median, MIN_NR_OBSERVED)

    epsilon_mean_k = epsilon_mean[epsilon_mean["nr_observed"] >= MIN_NR_OBSERVED]
    epsilon_median_k = epsilon_median[epsilon_median["nr_observed"] >= MIN_NR_OBSERVED]

    # Paired comparison population (k=3): precursors with a defined epsilon under BOTH aggregations
    # (i.e. quantified in both conditions). Used only for the mean-vs-median difference analysis.
    comparison_df = pd.merge(
        epsilon_mean_k[[PRECURSOR, "species", "epsilon"]].rename(columns={"epsilon": "epsilon_mean"}),
        epsilon_median_k[[PRECURSOR, "epsilon"]].rename(columns={"epsilon": "epsilon_median"}),
        on=PRECURSOR,
        how="inner",
    ).dropna()
    comparison_df["abs_diff"] = (comparison_df["epsilon_mean"] - comparison_df["epsilon_median"]).abs()
    comparison_df["tool"] = tool

    return {
        "tool": tool,
        "n_precursors": n_k3,
        "median_abs_epsilon_mean_based": median_abs_epsilon_mean_based_k3,
        "median_abs_epsilon_median_based": median_abs_epsilon_median_based_k3,
        # k=1 (unfiltered): matches the actual stored/displayed website values (see Validation section).
        "n_precursors_k1": n_k1,
        "median_abs_epsilon_mean_based_k1": median_abs_epsilon_mean_based_k1,
        "median_abs_epsilon_median_based_k1": median_abs_epsilon_median_based_k1,
        "epsilon_mean_k": epsilon_mean_k.assign(tool=tool),
        "epsilon_median_k": epsilon_median_k.assign(tool=tool),
        "comparison_df": comparison_df,
    }


_CACHE_KEYS = {"combined", "combined_epsilon_mean_k", "combined_epsilon_median_k", "result_metadata"}
_cache = None
if os.path.exists(CACHE_PATH) and not OVERWRITE_CACHE:
    with open(CACHE_PATH, "rb") as f:
        _loaded = pickle.load(f)
    if isinstance(_loaded, dict) and _CACHE_KEYS <= _loaded.keys():
        _cache = _loaded
    else:
        print(f"{CACHE_PATH} exists but isn't in the expected cache format (stale) -- recomputing.")

if _cache is not None:
    combined = _cache["combined"]
    combined_epsilon_mean_k = _cache["combined_epsilon_mean_k"]
    combined_epsilon_median_k = _cache["combined_epsilon_median_k"]
    result_metadata = _cache["result_metadata"]
    print(
        f"Loaded cached results from {CACHE_PATH}: {len(result_metadata)} submissions, "
        f"{len(combined)} paired precursor rows. Set OVERWRITE_CACHE=True above to refresh."
    )
else:
    result_metadata = {}  # submission id -> metadata dict (website-matching, both k=3 and k=1)
    all_epsilon_mean_k = {}
    all_epsilon_median_k = {}
    _comparison_dfs = {}  # submission id -> comparison_df (paired, k=3-filtered, non-NaN both sides)
    for sub_id, (tool, input_file, input_file_secondary) in SUBMISSION_INPUT_FILES.items():
        try:
            result = compare_tool(tool, input_file, input_file_secondary)
        except Exception as exc:
            print(f"{sub_id} ({tool}): failed to process ({exc!r}), skipped")
            continue
        if result["n_precursors"] == 0:
            print(f"{sub_id} ({tool}): no precursors left after single-species + nr_observed>={MIN_NR_OBSERVED} filtering, skipped")
            continue
        _comparison_dfs[sub_id] = result["comparison_df"].assign(submission_id=sub_id)
        result_metadata[sub_id] = {
            "tool": tool,
            "n_precursors": result["n_precursors"],
            "median_abs_epsilon_mean_based": result["median_abs_epsilon_mean_based"],
            "median_abs_epsilon_median_based": result["median_abs_epsilon_median_based"],
            "n_precursors_k1": result["n_precursors_k1"],
            "median_abs_epsilon_mean_based_k1": result["median_abs_epsilon_mean_based_k1"],
            "median_abs_epsilon_median_based_k1": result["median_abs_epsilon_median_based_k1"],
        }
        all_epsilon_mean_k[sub_id] = result["epsilon_mean_k"].assign(submission_id=sub_id)
        all_epsilon_median_k[sub_id] = result["epsilon_median_k"].assign(submission_id=sub_id)

    combined = pd.concat(_comparison_dfs.values(), ignore_index=True)
    combined_epsilon_mean_k = pd.concat(all_epsilon_mean_k.values(), ignore_index=True)
    combined_epsilon_median_k = pd.concat(all_epsilon_median_k.values(), ignore_index=True)

    with open(CACHE_PATH, "wb") as f:
        pickle.dump(
            {
                "combined": combined,
                "combined_epsilon_mean_k": combined_epsilon_mean_k,
                "combined_epsilon_median_k": combined_epsilon_median_k,
                "result_metadata": result_metadata,
            },
            f,
        )
    print(f"Saved results cache to {CACHE_PATH} ({len(result_metadata)} submissions).")

# Reconstructed from `combined` rather than cached separately, so the submission-summary cell
# below works identically whether `combined` came from the cache or from a fresh extraction.
all_results = {sub_id: df for sub_id, df in combined.groupby("submission_id")}

print(
    f"\n{combined_epsilon_mean_k.shape[0]} precursor rows (nr_observed >= {MIN_NR_OBSERVED}, website-matching count) "
    f"across {len(result_metadata)} submissions ({combined['tool'].nunique()} tools)"
)
print(f"{len(combined)} of those have a defined epsilon under both aggregations (used for the difference analysis)")


Loaded cached results from /mnt/c/Users/robbe/Work/Post-processing/analysis/all_datapoints_Astral.pkl: 64 submissions, 6676503 paired precursor rows. Set OVERWRITE_CACHE=True above to refresh.

6913361 precursor rows (nr_observed >= 3, website-matching count) across 64 submissions (6 tools)
6676503 of those have a defined epsilon under both aggregations (used for the difference analysis)


## Summary per submission

`n_precursors` and `median_abs_epsilon_*_based` come from `result_metadata` (the full k=3-filtered
population, matching the website's `nr_feature`/`median_abs_epsilon_eq_species` exactly). The
difference-analysis columns (`median_abs_diff`, `p90_abs_diff`, `max_abs_diff`, `pearson_r`) come
from `all_results` (the paired, both-conditions-quantified subset) — how far a single precursor's
accuracy estimate moves when switching aggregation, and the correlation between the two epsilon
series (1.0 = identical ranking/values).


In [25]:
submission_summary_rows = []
for sub_id, meta in result_metadata.items():
    comparison_df = all_results[sub_id]
    submission_summary_rows.append(
        {
            "id": sub_id,
            "tool": meta["tool"],
            "n_precursors": meta["n_precursors"],
            "median_abs_epsilon_mean_based": meta["median_abs_epsilon_mean_based"],
            "median_abs_epsilon_median_based": meta["median_abs_epsilon_median_based"],
            "n_precursors_k1": meta["n_precursors_k1"],
            "median_abs_epsilon_mean_based_k1": meta["median_abs_epsilon_mean_based_k1"],
            "median_abs_epsilon_median_based_k1": meta["median_abs_epsilon_median_based_k1"],
            "median_abs_diff": comparison_df["abs_diff"].median(),
            "p90_abs_diff": comparison_df["abs_diff"].quantile(0.9),
            "max_abs_diff": comparison_df["abs_diff"].max(),
            "pearson_r": comparison_df["epsilon_mean"].corr(comparison_df["epsilon_median"]),
        }
    )
submission_summary_df = (
    pd.DataFrame(submission_summary_rows).round(4).sort_values(["tool", "id"]).reset_index(drop=True)
)
submission_summary_df


,id,tool,n_precursors,median_abs_epsilon_mean_based,median_abs_epsilon_median_based,n_precursors_k1,median_abs_epsilon_mean_based_k1,median_abs_epsilon_median_based_k1,median_abs_diff,p90_abs_diff,max_abs_diff,pearson_r
0,AlphaDIA_20251017_120515,AlphaDIA,76323,0.2522,0.2572,91675,0.2639,0.2684,0.0360,0.1614,5.3253,0.9326
1,AlphaDIA_20251017_120858,AlphaDIA,108190,0.3706,0.3778,110569,0.3730,0.3800,0.0664,0.2349,4.0518,0.9436
2,AlphaDIA_20251017_121403,AlphaDIA,113175,0.3795,0.3826,115816,0.3823,0.3855,0.0673,0.2396,3.9621,0.9447
3,AlphaDIA_20260428_142928,AlphaDIA,84145,0.2474,0.2528,97484,0.2569,0.2620,0.0420,0.1680,5.4402,0.9497
4,AlphaDIA_20260708_174216,AlphaDIA,126853,0.3891,0.3927,128675,0.3908,0.3950,0.0723,0.2542,3.9582,0.9443
...,...,...,...,...,...,...,...,...,...,...,...,...
59,Spectronaut_20250707_115202,Spectronaut,107378,0.2622,0.2724,107882,0.2625,0.2728,0.0902,0.3979,4.4934,0.8993
60,Spectronaut_20250710_073727,Spectronaut,112238,0.2709,0.2849,112930,0.2712,0.2854,0.0919,0.4033,3.9237,0.9077
61,Spectronaut_20260226_080505,Spectronaut,117095,0.2803,0.2943,117658,0.2809,0.2948,0.0950,0.4142,3.6644,0.9136
62,Spectronaut_20260803_084932,Spectronaut,138352,0.3237,0.3324,141692,0.3267,0.3366,0.0997,0.4499,3.1007,0.9159


## Validation: does the recomputation match the real stored values?

`all_datapoints` already has the actual `median_abs_epsilon_eq_species` and `nr_feature` fields as
stored in each submission's public JSON (computed by the real `benchmarking()` pipeline when the
run was originally submitted). Checking those against `results.1.*`/`results.3.*` in the same JSON
shows that the flat, stored fields exactly equal **k=1** (unfiltered), not k=3 — even though the
current codebase's `default_cutoff_min_feature` is 3. So this notebook validates **both**: the k=1
numbers should match the stored values almost exactly (confirming the recomputation, including the
species-weighted aggregation, is correct), while the k=3 numbers are expected to differ (they
reflect what today's default threshold would produce, not what was actually stored for these
historical submissions).


In [26]:
stored = all_datapoints[["id", "nr_feature", "median_abs_epsilon_eq_species"]].rename(
    columns={"nr_feature": "nr_feature_stored", "median_abs_epsilon_eq_species": "median_abs_epsilon_eq_species_stored"}
)
validation_df = submission_summary_df.merge(stored, on="id", how="left")

validation_df["n_precursors_diff_k1"] = validation_df["n_precursors_k1"] - validation_df["nr_feature_stored"]
validation_df["metric_diff_k1"] = (
    validation_df["median_abs_epsilon_mean_based_k1"] - validation_df["median_abs_epsilon_eq_species_stored"]
)
validation_df["n_precursors_diff_k3"] = validation_df["n_precursors"] - validation_df["nr_feature_stored"]
validation_df["metric_diff_k3"] = (
    validation_df["median_abs_epsilon_mean_based"] - validation_df["median_abs_epsilon_eq_species_stored"]
)

print("k=1 (unfiltered) recomputation vs. stored production values -- should match almost exactly:")
print(
    validation_df[["id", "tool", "n_precursors_diff_k1", "metric_diff_k1"]].to_string(index=False)
)
print(f"\nMax |n_precursors_diff_k1| = {validation_df['n_precursors_diff_k1'].abs().max()}")
print(f"Max |metric_diff_k1|       = {validation_df['metric_diff_k1'].abs().max():.6f}")

print("\n\nk=3 (as requested) vs. stored production values -- expected to differ, since the stored")
print("values reflect k=1, not k=3:")
print(
    validation_df[["id", "tool", "n_precursors_diff_k3", "metric_diff_k3"]].to_string(index=False)
)
print(f"\nMax |n_precursors_diff_k3| = {validation_df['n_precursors_diff_k3'].abs().max()}")
print(f"Max |metric_diff_k3|       = {validation_df['metric_diff_k3'].abs().max():.4f}")


k=1 (unfiltered) recomputation vs. stored production values -- should match almost exactly:
                                     id                    tool  n_precursors_diff_k1  metric_diff_k1
               AlphaDIA_20251017_120515                AlphaDIA                     0    4.311220e-05
               AlphaDIA_20251017_120858                AlphaDIA                     0   -1.858267e-05
               AlphaDIA_20251017_121403                AlphaDIA                     0   -6.549269e-06
               AlphaDIA_20260428_142928                AlphaDIA                     0   -1.355599e-05
               AlphaDIA_20260708_174216                AlphaDIA                     0   -2.702001e-05
               AlphaDIA_20260709_073446                AlphaDIA                     0   -3.762338e-05
               AlphaDIA_20260709_104912                AlphaDIA                     0    7.068125e-06
               AlphaDIA_20260709_142829                AlphaDIA                     0   -2.9

## Summary per tool (pooled across all of that tool's submissions)

`n_precursors_pooled` and `median_abs_epsilon_*_based` are computed on `combined_epsilon_mean_k`/
`combined_epsilon_median_k` (the full k=3-filtered population, pooled across every submission of a
tool) — the tool-level analog of the website-matching per-submission numbers above. The
difference-analysis columns are computed on `combined` (the paired subset) as before. Tools with
many public submissions (e.g. MaxQuant, i2MassChroQ) are therefore backed by much more evidence
here than tools with only one (e.g. Sage, PEAKS, quantms, MSAngel, ProlineStudio at the time of
writing) — see `n_submissions`.


In [27]:
tool_summary_rows = []
for tool, group in combined.groupby("tool"):
    mean_group = combined_epsilon_mean_k[combined_epsilon_mean_k["tool"] == tool]
    median_group = combined_epsilon_median_k[combined_epsilon_median_k["tool"] == tool]
    tool_summary_rows.append(
        {
            "tool": tool,
            "n_submissions": group["submission_id"].nunique(),
            "n_precursors_pooled": len(mean_group),
            "median_abs_epsilon_mean_based": eq_species_metric(mean_group, "epsilon"),
            "median_abs_epsilon_median_based": eq_species_metric(median_group, "epsilon"),
            "median_abs_diff": group["abs_diff"].median(),
            "p90_abs_diff": group["abs_diff"].quantile(0.9),
            "pearson_r": group["epsilon_mean"].corr(group["epsilon_median"]),
        }
    )
tool_summary_df = (
    pd.DataFrame(tool_summary_rows).round(4).sort_values("n_submissions", ascending=False).reset_index(drop=True)
)
tool_summary_df


,tool,n_submissions,n_precursors_pooled,median_abs_epsilon_mean_based,median_abs_epsilon_median_based,median_abs_diff,p90_abs_diff,pearson_r
0,DIA-NN,32,3472684,0.2591,0.2714,0.0636,0.2537,0.9517
1,AlphaDIA,10,1115186,0.3545,0.3612,0.0621,0.2311,0.9469
2,PEAKS,8,904459,0.2368,0.2484,0.0620,0.2613,0.9415
3,Spectronaut,8,957658,0.2851,0.2973,0.0910,0.4028,0.9153
4,FragPipe (DIA-NN quant),5,441833,0.2311,0.2454,0.0631,0.2392,0.9438
5,MaxQuant,1,21541,0.2014,0.2098,0.0449,0.1643,0.9438


In [28]:
print("Overall (all tools pooled, species-weighted):")
print(f"  median |epsilon_mean|   (species-weighted) = {eq_species_metric(combined, 'epsilon_mean'):.4f}")
print(f"  median |epsilon_median| (species-weighted) = {eq_species_metric(combined, 'epsilon_median'):.4f}")
print(f"  median |epsilon_mean - epsilon_median|      = {combined['abs_diff'].median():.4f}")
print(f"  90th pct |epsilon_mean - epsilon_median|    = {combined['abs_diff'].quantile(0.9):.4f}")
print(f"  Pearson r(epsilon_mean, epsilon_median)     = {combined['epsilon_mean'].corr(combined['epsilon_median']):.4f}")

combined.groupby("species")["abs_diff"].median().rename("median_abs_diff_by_species")


Overall (all tools pooled, species-weighted):
  median |epsilon_mean|   (species-weighted) = 0.2690
  median |epsilon_median| (species-weighted) = 0.2812
  median |epsilon_mean - epsilon_median|      = 0.0662
  90th pct |epsilon_mean - epsilon_median|    = 0.2682
  Pearson r(epsilon_mean, epsilon_median)     = 0.9434


species
ECOLI    0.068199
HUMAN    0.065586
YEAST    0.067728
Name: median_abs_diff_by_species, dtype: float64

## Overview plot: accuracy vs. coverage (mean-based vs. median-based)

This mimics the website's actual main benchmarking plot (`LFQHYEPlotGenerator.plot_main_metric`)
with its **default settings**: x = median absolute epsilon in **Species-weighted** mode (average
of the per-species medians, equal weight regardless of precursor count — not "global", which pools
all species as one population), y = number of quantified precursors at
**k = MIN_NR_OBSERVED = 3** (`nr_observed >= 3`) — one point per public submission, colored by
tool (same palette as the manuscript figures), exactly like the real production plot.

Each submission is plotted twice — once with the production **mean-based** aggregation (circle)
and once with the **median-based** alternative (x) — connected by an arrow. Note that only the
x-position moves: which precursors pass the `nr_observed >= 3` filter (and therefore
`n_precursors`) does not depend on whether the per-condition intensity is aggregated with the mean
or the median, so the arrows are purely horizontal.


In [42]:
fig2 = go.Figure()
_ax = [0.225 * 0.95, 0.30 * 1.05]
_ay = [68000 * 0.95, 135000 * 1.05]

_tools_ordered = submission_summary_df["tool"].drop_duplicates().tolist()
for tool in _tools_ordered:
    color = _SOFTWARE_COLORS.get(tool, "#888888")
    sub_df = submission_summary_df[submission_summary_df["tool"] == tool]

    for _, row in sub_df.iterrows():
        fig2.add_annotation(
            x=row["median_abs_epsilon_median_based"],
            y=row["n_precursors"],
            ax=row["median_abs_epsilon_mean_based"],
            ay=row["n_precursors"],
            xref="x", yref="y", axref="x", ayref="y",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=1.2,
            arrowcolor=_hex_to_rgba(color, 0.7),
        )

    fig2.add_trace(
        go.Scatter(
            x=sub_df["median_abs_epsilon_mean_based"],
            y=sub_df["n_precursors"],
            mode="markers",
            marker=dict(color=color, size=9, symbol="circle", opacity=0.85, line=dict(color="white", width=0.5)),
            name=_display(tool),
            legendgroup=tool,
            hovertemplate=f"{_display(tool)} (mean-based)<br>x: %{{x:.3f}}<br>y: %{{y}}<extra></extra>",
        )
    )
    fig2.add_trace(
        go.Scatter(
            x=sub_df["median_abs_epsilon_median_based"],
            y=sub_df["n_precursors"],
            mode="markers",
            marker=dict(color=color, size=9, symbol="x", opacity=0.85),
            name=_display(tool),
            legendgroup=tool,
            showlegend=False,
            hovertemplate=f"{_display(tool)} (median-based)<br>x: %{{x:.3f}}<br>y: %{{y}}<extra></extra>",
        )
    )

# Second, marker-shape legend (circle = mean-based / production aggregation, x = median-based
# alternative). Dummy off-plot traces so only the legend entry renders, on a second native Plotly
# legend ("legend2") stacked below the tool-color legend.
_shape_legend_color = "#444444"
fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(color=_shape_legend_color, size=9, symbol="circle"),
        name="Mean-based (production)",
        legend="legend2",
    )
)
fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(color=_shape_legend_color, size=9, symbol="x"),
        name="Median-based (alternative)",
        legend="legend2",
    )
)

_metric_label = METRIC_AGG.capitalize()
fig2.update_xaxes(title_text=f"{_metric_label}QuantError({MIN_NR_OBSERVED})", **_AXIS_STYLE, range=_ax)
fig2.update_yaxes(title_text=f"Sensitivity({MIN_NR_OBSERVED})", **_AXIS_STYLE, range=_ay)
fig2.update_layout(
    **_BASE_LAYOUT,
    template="plotly_white",
    width=1000,
    height=760,
    legend=dict(
        orientation="h",
        x=0.5, xanchor="center",
        y=-0.14, yanchor="top",
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(size=18)
    ),
    legend2=dict(
        orientation="h",
        x=0.5, xanchor="center",
        y=-0.24, yanchor="top",
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(size=18)
    ),
    margin=dict(l=70, r=40, t=40, b=160),
)
fig2.show()

shift = (submission_summary_df["median_abs_epsilon_median_based"] - submission_summary_df["median_abs_epsilon_mean_based"]).abs()
print("Horizontal shift in the reported accuracy metric per submission (median-based - mean-based, absolute):")
print(
    submission_summary_df[["id", "tool"]]
    .assign(metric_shift=shift.round(4))
    .sort_values("metric_shift", ascending=False)
    .to_string(index=False)
)

fig2.write_image("mean_vs_median_epsilon.png", scale=2)
fig2.write_image("mean_vs_median_epsilon.svg")


Horizontal shift in the reported accuracy metric per submission (median-based - mean-based, absolute):
                                     id                    tool  metric_shift
            Spectronaut_20260803_090414             Spectronaut        0.0177
                 DIA-NN_20260805_082913                  DIA-NN        0.0167
FragPipe (DIA-NN quant)_20260709_073103 FragPipe (DIA-NN quant)        0.0156
                 DIA-NN_20251027_114513                  DIA-NN        0.0155
FragPipe (DIA-NN quant)_20260428_144834 FragPipe (DIA-NN quant)        0.0149
FragPipe (DIA-NN quant)_20251027_124759 FragPipe (DIA-NN quant)        0.0148
                 DIA-NN_20251027_120524                  DIA-NN        0.0147
                 DIA-NN_20251027_120754                  DIA-NN        0.0147
                  PEAKS_20250605_172158                   PEAKS        0.0146
                 DIA-NN_20260107_154840                  DIA-NN        0.0144
                 DIA-NN_20251113_143358

## Notes and caveats

- This notebook now matches the website's default main-plot settings exactly: **Median**
  aggregation, **Species-weighted** mode (`median_abs_epsilon_eq_species`, the average of the
  per-species medians with equal weight — not `_global`, which pools all species as one
  population), and **k = MIN_NR_OBSERVED = 3** (only precursors observed in >= 3 replicate raw
  files count). See the "Validation" section above for a direct comparison of the mean-based
  recomputation against the actual stored `nr_feature`/`median_abs_epsilon_eq_species` values from
  each submission's public JSON.
- This uses **every real, public submission** for `quant_lfq_DDA_ion_QExactive` at the time this
  notebook was run, downloaded live from `Proteobench/Results_quant_ion_DDA` and the public data
  server (or from the local cache described above, if present). Results will differ on a re-run if
  new submissions are added and `OVERWRITE_CACHE` is set. Tools are backed by very different
  numbers of public submissions (e.g. 15 for MaxQuant and 10 for i2MassChroQ vs. 1 each for
  Sage/PEAKS/quantms/MSAngel/ProlineStudio at the time of writing, see `n_submissions` in the
  per-tool summary) — pooled per-tool statistics should be read with that imbalance in mind, and are
  not a substitute for looking at the per-submission table.
- A submission is silently skipped if its raw data zip is unavailable on the public server, has no
  `input_file.*`, or fails to parse/produces no precursors passing the single-species +
  `nr_observed >= 3` filtering.
- The correlation between the two epsilon series is high for every submission, so median-based
  aggregation would not change the qualitative ranking of tools, but per-precursor epsilon can shift
  by a few tenths of a log2 unit, and the `median_abs_epsilon_*` accuracy metric itself moves
  measurably per submission (see the "Horizontal shift" ranking under the overview plot).
- Only `log_Intensity_mean` / `log2_A_vs_B` (and therefore `epsilon`) is swapped to median here.
  `CV` (which uses `Intensity_std`/`Intensity_mean`) and the precision epsilons (which use the
  empirical median/mean of `log2_A_vs_B` across precursors, already computed both ways in
  `compute_epsilon`) are unaffected by this specific change.
- The overview plot is Plotly, styled to match the ProteoBench manuscript figures
  (`_SOFTWARE_COLORS`/`_SPECIES_COLORS`/`_BASE_LAYOUT`/`_AXIS_STYLE` defined in the "ProteoBench-
  styled Plotly helpers" cell near the top). If a tool appears that isn't in `_SOFTWARE_COLORS`,
  it falls back to a neutral gray (`#888888`) rather than erroring.
- The earlier per-precursor scatter (mean-based vs. median-based epsilon, colored by species) and
  its companion per-tool boxplot were removed: plotting all `combined` rows (millions of points)
  as an interactive Plotly figure embeds every point as JSON in the notebook file, which is what
  previously made this notebook balloon to 200+ MB and become unresponsive to open/edit. The
  aggregate statistics they illustrated (`median_abs_diff`, `p90_abs_diff`, `pearson_r`, and the
  per-species breakdown) are still computed above, in the tool-summary and overall-stats cells.
